# Module 7: Sampling & Inference

Sampling is the payoff of training a diffusion model: starting from pure noise, we iteratively denoise to produce realistic images. This module covers the core sampling algorithms you need to know for interviews, from the original DDPM reverse process to accelerated DDIM sampling and the probability flow ODE perspective.

**Learning Objectives**
- Implement DDPM sampling (Algorithm 2 from [Ho et al. 2020](https://arxiv.org/abs/2006.11239)) from scratch
- Implement DDIM sampling ([Song et al. 2020](https://arxiv.org/abs/2010.02502)) with tunable stochasticity (eta)
- Accelerate sampling by using fewer timesteps with DDIM
- Understand how noise schedules interact with inference
- Connect sampling to the probability flow ODE
- Build a complete, production-quality sampling pipeline

**Estimated time:** 2--3 hours

**Key References:**
- DDPM -- Ho et al. 2020: https://arxiv.org/abs/2006.11239 (Algorithm 2)
- DDIM -- Song et al. 2020: https://arxiv.org/abs/2010.02502
- Progressive Distillation -- Salimans & Ho 2022: https://arxiv.org/abs/2202.00512

In [ ]:
import sys
import os
import time
import math
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Shared utilities
sys.path.insert(0, '.')
from utils.schedule import cosine_schedule, get_schedule
from utils.visualization import show_images, denormalize, plot_loss_curve, show_denoising_trajectory, set_style
from utils.data import get_mnist_dataloader, get_device

# Reproducibility
torch.manual_seed(42)

# Device setup
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

set_style()

## Model Definition and Checkpoint Loading

We import the **canonical UNet from `utils/unet.py`** — the same architecture used in Modules 4 and 6.
This ensures checkpoint compatibility: the model trained in Module 6 loads directly here.

If no checkpoint is found, we train a quick model as a fallback (using the same canonical UNet).

In [ ]:
# Import the canonical UNet (same architecture used in Modules 4 and 6)
from utils.unet import UNet

In [ ]:
# --------------------------------------------------------------------------
# Load checkpoint or train a quick model
# --------------------------------------------------------------------------

T = 1000
schedule = get_schedule("cosine", T=T)
# Move schedule tensors to device once
for key in schedule:
    schedule[key] = schedule[key].to(device)

model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)

checkpoint_path = "checkpoints/ddpm_mnist.pt"
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    state = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    elif isinstance(state, dict) and 'ema_shadow' in state:
        # Load EMA weights if available (better quality)
        ema_shadow = state['ema_shadow']
        for name, param in model.named_parameters():
            if name in ema_shadow:
                param.data.copy_(ema_shadow[name])
    else:
        model.load_state_dict(state)
    print("Checkpoint loaded successfully.")
else:
    print(f"No checkpoint found at {checkpoint_path}. Training a quick model...")
    
    # Quick training: 2000 steps on MNIST
    dataloader = get_mnist_dataloader(batch_size=64, image_size=28)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
    
    model.train()
    step = 0
    max_steps = 2000
    losses = []
    
    while step < max_steps:
        for batch, _ in dataloader:
            if step >= max_steps:
                break
            batch = batch.to(device)  # (B, 1, 28, 28)
            
            # Sample random timesteps
            t = torch.randint(0, T, (batch.shape[0],), device=device)  # (B,)
            
            # Forward diffusion: add noise
            noise = torch.randn_like(batch)  # (B, 1, 28, 28)
            sqrt_alpha_bar = schedule['sqrt_alphas_cumprod'][t][:, None, None, None]
            sqrt_one_minus = schedule['sqrt_one_minus_alphas_cumprod'][t][:, None, None, None]
            x_t = sqrt_alpha_bar * batch + sqrt_one_minus * noise
            
            # Predict noise
            noise_pred = model(x_t, t)  # (B, 1, 28, 28)
            loss = F.mse_loss(noise_pred, noise)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            losses.append(loss.item())
            step += 1
            if step % 500 == 0:
                print(f"  Step {step}/{max_steps}, Loss: {loss.item():.4f}")
    
    print(f"Training complete. Final loss: {losses[-1]:.4f}")
    plot_loss_curve(losses, window=50, title="Quick Training Loss")

model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

---
## 7.1 -- DDPM Sampling (Algorithm 2)

The DDPM reverse process from [Ho et al. 2020](https://arxiv.org/abs/2006.11239), Algorithm 2:

1. Start with $x_T \sim \mathcal{N}(0, I)$
2. For $t = T, T-1, \ldots, 1$:
   - Predict noise: $\epsilon_\theta = \text{model}(x_t, t)$
   - Compute posterior mean: $\mu_\theta = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta \right)$
   - Sample: $x_{t-1} = \mu_\theta + \sigma_t z$, where $z \sim \mathcal{N}(0, I)$ for $t > 1$, $z = 0$ for $t = 1$

Two common choices for $\sigma_t$:

| Variance choice | Formula | When to use |
|:---|:---|:---|
| DDPM default | $\sigma_t^2 = \beta_t$ | Standard choice, simpler |
| Posterior variance | $\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \cdot \beta_t$ | Tighter bound, sometimes better quality |

The full loop requires T = 1000 forward passes through the model -- this is why sampling is slow and why accelerated methods (Section 7.4) matter.

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: DDPM Sampling
# --------------------------------------------------------------------------

@torch.no_grad()
def ddpm_sample(
    model: nn.Module,
    shape: Tuple[int, ...],
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    variance_type: str = "beta",
    track_trajectory: bool = False,
    trajectory_steps: Optional[List[int]] = None,
) -> Tuple[torch.Tensor, List[torch.Tensor]]:
    """DDPM sampling -- Algorithm 2 from Ho et al. 2020.
    
    Args:
        model: Trained noise prediction model.
        shape: (B, C, H, W) shape of samples to generate.
        schedule: Pre-computed schedule dictionary.
        T: Number of diffusion timesteps.
        variance_type: 'beta' for sigma_t^2=beta_t, 'posterior' for tilde_beta_t.
        track_trajectory: If True, save intermediate states.
        trajectory_steps: Which timesteps to save (descending order).
    
    Returns:
        (samples, trajectory) -- final samples and list of intermediate images.
    """
    device = next(model.parameters()).device
    
    # Step 1: Start from pure noise
    x_t = torch.randn(shape, device=device)  # (B, C, H, W)
    
    trajectory = []
    if trajectory_steps is None:
        trajectory_steps = [T - 1, 750, 500, 250, 100, 50, 10, 0]
    
    # Step 2: Reverse process t = T-1, T-2, ..., 0
    for t_val in reversed(range(T)):
        t_batch = torch.full((shape[0],), t_val, device=device, dtype=torch.long)  # (B,)
        
        # Predict noise
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        # Extract schedule values for this timestep
        beta_t = schedule['betas'][t_val]  # scalar
        alpha_t = schedule['alphas'][t_val]  # scalar
        alpha_bar_t = schedule['alphas_cumprod'][t_val]  # scalar
        sqrt_recip_alpha_t = schedule['sqrt_recip_alphas'][t_val]  # scalar
        sqrt_one_minus_alpha_bar_t = schedule['sqrt_one_minus_alphas_cumprod'][t_val]  # scalar
        
        # Compute posterior mean
        # mu_theta = (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1-alpha_bar_t) * eps_theta)
        mu_theta = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alpha_bar_t * eps_theta)  # (B, C, H, W)
        
        # Choose variance
        if variance_type == "beta":
            sigma_t_sq = beta_t  # DDPM default
        elif variance_type == "posterior":
            sigma_t_sq = schedule['posterior_variance'][t_val]  # tilde_beta_t
        else:
            raise ValueError(f"Unknown variance_type: {variance_type}")
        
        # Sample x_{t-1}
        if t_val > 0:
            z = torch.randn_like(x_t)  # (B, C, H, W)
            x_t = mu_theta + torch.sqrt(sigma_t_sq) * z  # (B, C, H, W)
        else:
            x_t = mu_theta  # No noise at final step
        
        # Track trajectory
        if track_trajectory and t_val in trajectory_steps:
            trajectory.append(x_t[0:1].clone())  # save first sample
    
    return x_t, trajectory  # (B, C, H, W), List[(1, C, H, W)]

In [ ]:
# Generate samples and visualize the denoising trajectory
torch.manual_seed(42)

trajectory_steps = [999, 750, 500, 250, 100, 50, 10, 0]

samples, trajectory = ddpm_sample(
    model, shape=(16, 1, 28, 28), schedule=schedule, T=T,
    track_trajectory=True, trajectory_steps=trajectory_steps,
)

print(f"Sample shape: {samples.shape}, range: [{samples.min():.2f}, {samples.max():.2f}]")

# Show denoising trajectory for one sample
show_denoising_trajectory(trajectory, timesteps=trajectory_steps)

# Show full grid of generated samples
show_images(samples, nrow=4, title="DDPM Samples (1000 steps)")

### Exercise 7.1: DDPM Sampling with Both Variance Choices

Generate a grid of samples using both variance choices (`beta` and `posterior`). Compare visually and verify the outputs resemble MNIST digits.

In [ ]:
# Exercise 7.1: Your code here
# 1. Generate 16 samples with variance_type='beta'
# 2. Generate 16 samples with variance_type='posterior' (same initial noise)
# 3. Display both grids side by side
# 4. Comment on any visual differences

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Generate with both variance types, same seed for fair comparison
torch.manual_seed(123)
initial_noise = torch.randn(16, 1, 28, 28, device=device)  # (16, 1, 28, 28)

# We need a version that accepts initial noise
@torch.no_grad()
def ddpm_sample_from_noise(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    variance_type: str = "beta",
) -> torch.Tensor:
    """DDPM sampling starting from a given noise tensor."""
    device = next(model.parameters()).device
    x_t = x_T.clone()  # (B, C, H, W)
    
    for t_val in reversed(range(T)):
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        beta_t = schedule['betas'][t_val]
        sqrt_recip_alpha_t = schedule['sqrt_recip_alphas'][t_val]
        sqrt_one_minus_alpha_bar_t = schedule['sqrt_one_minus_alphas_cumprod'][t_val]
        
        mu_theta = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alpha_bar_t * eps_theta)
        
        sigma_t_sq = schedule['betas'][t_val] if variance_type == 'beta' else schedule['posterior_variance'][t_val]
        
        if t_val > 0:
            z = torch.randn_like(x_t)
            x_t = mu_theta + torch.sqrt(sigma_t_sq) * z
        else:
            x_t = mu_theta
    
    return x_t

samples_beta = ddpm_sample_from_noise(model, initial_noise, schedule, T=T, variance_type='beta')
samples_posterior = ddpm_sample_from_noise(model, initial_noise, schedule, T=T, variance_type='posterior')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Display beta variance samples
import torchvision.utils as vutils
grid_beta = vutils.make_grid(denormalize(samples_beta), nrow=4, padding=2)
axes[0].imshow(grid_beta.permute(1, 2, 0).cpu().numpy())
axes[0].set_title("Variance: beta_t (DDPM default)", fontsize=12)
axes[0].axis('off')

grid_post = vutils.make_grid(denormalize(samples_posterior), nrow=4, padding=2)
axes[1].imshow(grid_post.permute(1, 2, 0).cpu().numpy())
axes[1].set_title("Variance: posterior (tilde_beta_t)", fontsize=12)
axes[1].axis('off')

plt.suptitle("DDPM Sampling: Two Variance Choices", fontsize=14)
plt.tight_layout()
plt.show()

# Note: Both produce valid samples. The stochastic noise at each step means
# results diverge even from the same x_T. The posterior variance is theoretically
# tighter (it's the exact posterior when the model is perfect), but in practice
# the difference is often subtle on well-trained models.

---
## 7.2 -- Why We Add Noise During Sampling

It seems counterintuitive: we trained the model to **remove** noise, yet during sampling we **add** noise at every step (the $\sigma_t z$ term). Why?

The reverse process is not simply denoising -- it is **sampling from a learned conditional distribution** $p_\theta(x_{t-1} | x_t)$, which is a Gaussian. The mean $\mu_\theta$ gives the most likely $x_{t-1}$, but sampling from the full Gaussian provides:

- **Error correction**: The stochastic noise lets the model recover from accumulated prediction errors. Without it, small errors in early steps compound.
- **Diversity**: Different random draws produce different samples from the same initial $x_T$.
- **Faithful distribution**: The theory guarantees convergence to $p_\text{data}$ only with the correct noise injection.

Setting $\sigma_t = 0$ (no noise) gives a **deterministic** mapping from $x_T$ to $x_0$ -- this is the DDIM limit (eta=0), which trades diversity for reproducibility.

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: Stochastic vs Deterministic Sampling
# --------------------------------------------------------------------------

@torch.no_grad()
def sample_with_noise_control(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    add_noise: bool = True,
) -> torch.Tensor:
    """Sample with or without the stochastic noise term."""
    device = next(model.parameters()).device
    x_t = x_T.clone()
    
    for t_val in reversed(range(T)):
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        beta_t = schedule['betas'][t_val]
        sqrt_recip_alpha_t = schedule['sqrt_recip_alphas'][t_val]
        sqrt_one_minus_alpha_bar_t = schedule['sqrt_one_minus_alphas_cumprod'][t_val]
        
        mu_theta = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alpha_bar_t * eps_theta)
        
        if t_val > 0 and add_noise:
            z = torch.randn_like(x_t)
            x_t = mu_theta + torch.sqrt(beta_t) * z
        else:
            x_t = mu_theta
    
    return x_t

# Same starting noise
torch.manual_seed(42)
shared_noise = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

# With noise (stochastic) -- run 3 times to show diversity
stochastic_runs = []
for i in range(3):
    torch.manual_seed(i * 100)  # Different internal randomness
    result = sample_with_noise_control(model, shared_noise, schedule, T=T, add_noise=True)
    stochastic_runs.append(result)

# Without noise (deterministic) -- run 3 times, should be identical
deterministic_runs = []
for i in range(3):
    torch.manual_seed(i * 100)  # Seed doesn't matter since no randomness is used
    result = sample_with_noise_control(model, shared_noise, schedule, T=T, add_noise=False)
    deterministic_runs.append(result)

# Display results
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for i in range(3):
    grid = vutils.make_grid(denormalize(stochastic_runs[i]), nrow=4, padding=2)
    axes[0, i].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[0, i].set_title(f"Stochastic run {i+1}", fontsize=11)
    axes[0, i].axis('off')

for i in range(3):
    grid = vutils.make_grid(denormalize(deterministic_runs[i]), nrow=4, padding=2)
    axes[1, i].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[1, i].set_title(f"Deterministic run {i+1}", fontsize=11)
    axes[1, i].axis('off')

plt.suptitle("Same x_T: Stochastic produces diverse outputs; Deterministic is repeatable", fontsize=13)
plt.tight_layout()
plt.show()

# Verify deterministic runs are identical
diff = (deterministic_runs[0] - deterministic_runs[1]).abs().max().item()
print(f"Max pixel difference between deterministic runs: {diff:.6f}")
print("(Should be 0.0 -- same x_T with no noise always yields the same x_0)")

---
## 7.3 -- DDIM: The Deterministic ODE Formulation

[DDIM (Song et al. 2020)](https://arxiv.org/abs/2010.02502) generalizes DDPM by introducing a parameter $\eta \in [0, 1]$ that controls stochasticity:

| Setting | Behavior |
|:---|:---|
| $\eta = 1$ | Equivalent to DDPM (full stochastic) |
| $\eta = 0$ | Fully deterministic (probability flow ODE) |
| $0 < \eta < 1$ | Interpolation between the two extremes |

**DDIM update rule:**

1. Predict clean image: $\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}$

2. Compute noise coefficient: $\sigma_t = \eta \cdot \sqrt{\frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}} \cdot \sqrt{1 - \frac{\bar{\alpha}_t}{\bar{\alpha}_{t-1}}}$

3. Compute direction pointing to $x_t$: $\text{dir} = \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \epsilon_\theta$

4. Sample: $x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \cdot \hat{x}_0 + \text{dir} + \sigma_t \cdot z$

**Key insight:** DDIM defines a *non-Markovian* forward process that has the **same marginals** $q(x_t | x_0)$ as DDPM. This means a model trained with DDPM can be used directly with DDIM sampling -- no retraining needed. When $\eta = 0$, the sampling follows the probability flow ODE, which maps noise to images deterministically.

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: DDIM Sampling
# --------------------------------------------------------------------------

@torch.no_grad()
def ddim_sample(
    model: nn.Module,
    shape: Tuple[int, ...],
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    eta: float = 0.0,
    num_steps: Optional[int] = None,
    track_trajectory: bool = False,
    trajectory_steps: Optional[List[int]] = None,
    x_T: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, List[torch.Tensor]]:
    """DDIM sampling with tunable stochasticity.
    
    Args:
        model: Trained noise prediction model.
        shape: (B, C, H, W) sample shape.
        schedule: Pre-computed schedule dictionary.
        T: Total number of training timesteps.
        eta: Stochasticity parameter. 0=deterministic, 1=DDPM.
        num_steps: Number of sampling steps (None=T for full sampling).
        track_trajectory: Whether to save intermediate states.
        trajectory_steps: Indices into the timestep sequence to save.
        x_T: Optional initial noise (for reproducibility comparisons).
    
    Returns:
        (samples, trajectory)
    """
    device = next(model.parameters()).device
    
    # Create timestep subsequence
    if num_steps is None:
        num_steps = T
    # Uniform spacing: e.g., for 50 steps over T=1000 -> [0, 20, 40, ..., 980]
    step_indices = torch.linspace(0, T - 1, num_steps, dtype=torch.long)  # (num_steps,)
    timesteps = step_indices.flip(0)  # Reverse: high to low
    
    # Start from noise
    if x_T is not None:
        x_t = x_T.clone()
    else:
        x_t = torch.randn(shape, device=device)  # (B, C, H, W)
    
    trajectory = []
    alphas_cumprod = schedule['alphas_cumprod']  # (T,)
    
    for i in range(len(timesteps)):
        t_val = timesteps[i].item()
        t_batch = torch.full((shape[0],), t_val, device=device, dtype=torch.long)  # (B,)
        
        # Predict noise
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        # Current and previous alpha_bar
        alpha_bar_t = alphas_cumprod[t_val]  # scalar
        if i < len(timesteps) - 1:
            t_prev = timesteps[i + 1].item()
            alpha_bar_t_prev = alphas_cumprod[t_prev]  # scalar
        else:
            alpha_bar_t_prev = torch.tensor(1.0, device=device)  # alpha_bar_0 = 1
        
        # Step 1: Predict x_0
        predicted_x0 = (x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)  # (B, C, H, W)
        predicted_x0 = predicted_x0.clamp(-1.0, 1.0)  # Clamp for stability
        
        # Step 2: Compute sigma_t
        # sigma_t = eta * sqrt((1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)) * sqrt(1 - alpha_bar_t / alpha_bar_{t-1})
        sigma_t = eta * torch.sqrt(
            (1.0 - alpha_bar_t_prev) / (1.0 - alpha_bar_t + 1e-8)
        ) * torch.sqrt(
            1.0 - alpha_bar_t / (alpha_bar_t_prev + 1e-8)
        )
        
        # Step 3: Direction pointing to x_t
        direction = torch.sqrt(
            torch.clamp(1.0 - alpha_bar_t_prev - sigma_t ** 2, min=0.0)
        ) * eps_theta  # (B, C, H, W)
        
        # Step 4: Combine
        x_t = torch.sqrt(alpha_bar_t_prev) * predicted_x0 + direction  # (B, C, H, W)
        
        if sigma_t > 0:
            z = torch.randn_like(x_t)  # (B, C, H, W)
            x_t = x_t + sigma_t * z
        
        # Track trajectory
        if track_trajectory and trajectory_steps is not None and i in trajectory_steps:
            trajectory.append(x_t[0:1].clone())
    
    return x_t, trajectory

In [ ]:
# Demonstrate DDIM with different eta values
torch.manual_seed(42)
shared_noise = torch.randn(8, 1, 28, 28, device=device)  # (8, 1, 28, 28)

eta_values = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(eta_values), figsize=(3 * len(eta_values), 4))

for idx, eta in enumerate(eta_values):
    samples, _ = ddim_sample(
        model, shape=(8, 1, 28, 28), schedule=schedule, T=T,
        eta=eta, x_T=shared_noise.clone(),
    )
    grid = vutils.make_grid(denormalize(samples), nrow=4, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[idx].set_title(f"eta={eta}", fontsize=12)
    axes[idx].axis('off')

plt.suptitle("DDIM: eta controls stochasticity (0=deterministic, 1=DDPM)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Verify: eta=0 with same x_T always gives same result
torch.manual_seed(42)
fixed_noise = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

result_a, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, x_T=fixed_noise.clone())
result_b, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, x_T=fixed_noise.clone())

max_diff = (result_a - result_b).abs().max().item()
print(f"DDIM eta=0: max difference between two runs from same x_T: {max_diff:.8f}")
print("This confirms deterministic sampling: same noise -> same image.")

### Exercise 7.3: Implement DDIM and Sweep Eta

Implement your own DDIM sampler from scratch (without looking at the worked example above), then sweep eta from 0 to 1 in increments of 0.2. For each eta value, generate 8 samples from the same initial noise and display the grids. Discuss how diversity changes with eta.

In [ ]:
# Exercise 7.3: Your code here
# 1. Implement ddim_sample_exercise(model, x_T, schedule, T, eta) -> samples
# 2. Generate with eta = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0] from same x_T
# 3. Display grids and discuss diversity

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

@torch.no_grad()
def ddim_sample_exercise(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    eta: float = 0.0,
) -> torch.Tensor:
    """DDIM sampling implementation (exercise version)."""
    device = next(model.parameters()).device
    x_t = x_T.clone()  # (B, C, H, W)
    alphas_cumprod = schedule['alphas_cumprod']  # (T,)
    
    for t_val in reversed(range(T)):
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)
        
        # Predict noise
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        alpha_bar_t = alphas_cumprod[t_val]
        alpha_bar_prev = alphas_cumprod[t_val - 1] if t_val > 0 else torch.tensor(1.0, device=device)
        
        # Predict x_0
        pred_x0 = (x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)  # (B, C, H, W)
        pred_x0 = pred_x0.clamp(-1.0, 1.0)
        
        # Compute sigma
        sigma = eta * torch.sqrt(
            (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t + 1e-8)
        ) * torch.sqrt(1.0 - alpha_bar_t / (alpha_bar_prev + 1e-8))
        
        # Direction
        direction = torch.sqrt(torch.clamp(1.0 - alpha_bar_prev - sigma**2, min=0.0)) * eps_theta
        
        # Update
        x_t = torch.sqrt(alpha_bar_prev) * pred_x0 + direction  # (B, C, H, W)
        if sigma > 0 and t_val > 0:
            x_t = x_t + sigma * torch.randn_like(x_t)
    
    return x_t

# Sweep eta
torch.manual_seed(42)
fixed_noise = torch.randn(8, 1, 28, 28, device=device)  # (8, 1, 28, 28)

eta_sweep = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
fig, axes = plt.subplots(1, len(eta_sweep), figsize=(3 * len(eta_sweep), 4))

for idx, eta in enumerate(eta_sweep):
    samples = ddim_sample_exercise(model, fixed_noise.clone(), schedule, T=T, eta=eta)
    grid = vutils.make_grid(denormalize(samples), nrow=4, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[idx].set_title(f"eta={eta}", fontsize=11)
    axes[idx].axis('off')

plt.suptitle("DDIM Eta Sweep: Deterministic -> Stochastic", fontsize=13)
plt.tight_layout()
plt.show()

# Observation: At eta=0 the outputs are identical across runs (deterministic).
# As eta increases, stochastic noise is injected at each step, producing more
# diverse (but noisier) outputs. At eta=1, it is equivalent to DDPM.

---
## 7.4 -- Fewer Steps: Accelerated Sampling

DDPM requires T=1000 sequential model evaluations -- impractical for real-time applications. DDIM's key practical advantage is that it allows **timestep sub-selection**: instead of iterating over all 1000 steps, we select a subset and skip the rest.

| Steps | Speedup | Typical Quality |
|:---:|:---:|:---|
| 1000 | 1x (baseline) | Best |
| 200 | 5x | Near-best |
| 50 | 20x | Good, slight degradation |
| 20 | 50x | Acceptable, visible artifacts |
| 10 | 100x | Recognizable but low quality |

**Subset selection strategies:**
- **Uniform spacing** (most common): `[0, 20, 40, ..., 980]` for 50 steps
- **Quadratic spacing**: More steps near t=0 where fine details emerge
- **Custom / learned**: Optimized for specific models (see [Progressive Distillation, Salimans & Ho 2022](https://arxiv.org/abs/2202.00512))

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: Accelerated Sampling with Fewer Steps
# --------------------------------------------------------------------------

torch.manual_seed(42)
fixed_noise = torch.randn(8, 1, 28, 28, device=device)  # (8, 1, 28, 28)

step_counts = [1000, 200, 50, 20, 10]
timings = {}
results = {}

for n_steps in step_counts:
    start_time = time.time()
    samples, _ = ddim_sample(
        model, shape=(8, 1, 28, 28), schedule=schedule, T=T,
        eta=0.0, num_steps=n_steps, x_T=fixed_noise.clone(),
    )
    elapsed = time.time() - start_time
    timings[n_steps] = elapsed
    results[n_steps] = samples
    print(f"  {n_steps:>5} steps: {elapsed:.2f}s")

# Visual comparison
fig, axes = plt.subplots(1, len(step_counts), figsize=(3 * len(step_counts), 4))

for idx, n_steps in enumerate(step_counts):
    grid = vutils.make_grid(denormalize(results[n_steps]), nrow=4, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[idx].set_title(f"{n_steps} steps\n({timings[n_steps]:.1f}s)", fontsize=11)
    axes[idx].axis('off')

plt.suptitle("DDIM: Quality vs Speed Tradeoff", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Plot wall-clock time vs number of steps
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(list(timings.keys()), list(timings.values()), 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Number of Sampling Steps', fontsize=12)
ax.set_ylabel('Wall-Clock Time (seconds)', fontsize=12)
ax.set_title('Sampling Time vs Number of Steps', fontsize=13)
ax.set_xscale('log')
for n, t in timings.items():
    ax.annotate(f"{t:.1f}s", (n, t), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### Exercise 7.4: Timestep Sub-Selection for DDIM

Implement both uniform and quadratic timestep spacing for DDIM. Generate samples at 50 steps with each strategy and compare. Then find the minimum number of steps that still produces recognizable MNIST digits.

In [ ]:
# Exercise 7.4: Your code here
# 1. Implement uniform and quadratic timestep spacing functions
# 2. Compare 50-step DDIM with each spacing strategy
# 3. Find minimum steps for acceptable quality

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def uniform_timesteps(T: int, num_steps: int) -> torch.Tensor:
    """Uniformly spaced timestep subsequence."""
    return torch.linspace(0, T - 1, num_steps, dtype=torch.long)


def quadratic_timesteps(T: int, num_steps: int) -> torch.Tensor:
    """Quadratic spacing: more steps near t=0 where fine details are resolved."""
    t = torch.linspace(0, 1, num_steps) ** 2  # quadratic in [0, 1]
    return (t * (T - 1)).long()


@torch.no_grad()
def ddim_sample_custom_steps(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    timestep_seq: torch.Tensor,
    eta: float = 0.0,
) -> torch.Tensor:
    """DDIM sampling with a custom timestep sequence."""
    device = next(model.parameters()).device
    x_t = x_T.clone()
    alphas_cumprod = schedule['alphas_cumprod']
    
    # Reverse the sequence (high to low)
    timesteps = timestep_seq.flip(0)
    
    for i in range(len(timesteps)):
        t_val = timesteps[i].item()
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)
        
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        alpha_bar_t = alphas_cumprod[t_val]
        if i < len(timesteps) - 1:
            alpha_bar_prev = alphas_cumprod[timesteps[i + 1].item()]
        else:
            alpha_bar_prev = torch.tensor(1.0, device=device)
        
        pred_x0 = ((x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)).clamp(-1, 1)
        
        sigma = eta * torch.sqrt(
            (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t + 1e-8)
        ) * torch.sqrt(1.0 - alpha_bar_t / (alpha_bar_prev + 1e-8))
        
        direction = torch.sqrt(torch.clamp(1.0 - alpha_bar_prev - sigma**2, min=0.0)) * eps_theta
        x_t = torch.sqrt(alpha_bar_prev) * pred_x0 + direction
        
        if sigma > 0:
            x_t = x_t + sigma * torch.randn_like(x_t)
    
    return x_t

# Compare uniform vs quadratic at 50 steps
torch.manual_seed(42)
fixed_noise = torch.randn(8, 1, 28, 28, device=device)

uniform_seq = uniform_timesteps(T, 50)
quadratic_seq = quadratic_timesteps(T, 50)

samples_uniform = ddim_sample_custom_steps(model, fixed_noise.clone(), schedule, uniform_seq, eta=0.0)
samples_quadratic = ddim_sample_custom_steps(model, fixed_noise.clone(), schedule, quadratic_seq, eta=0.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
grid_u = vutils.make_grid(denormalize(samples_uniform), nrow=4, padding=2)
axes[0].imshow(grid_u.permute(1, 2, 0).cpu().numpy())
axes[0].set_title("Uniform spacing (50 steps)", fontsize=12)
axes[0].axis('off')

grid_q = vutils.make_grid(denormalize(samples_quadratic), nrow=4, padding=2)
axes[1].imshow(grid_q.permute(1, 2, 0).cpu().numpy())
axes[1].set_title("Quadratic spacing (50 steps)", fontsize=12)
axes[1].axis('off')

plt.suptitle("Timestep Spacing Strategies", fontsize=13)
plt.tight_layout()
plt.show()

# Find minimum steps
print("\nMinimum steps search:")
for n in [5, 8, 10, 15, 20, 30, 50]:
    seq = uniform_timesteps(T, n)
    s = ddim_sample_custom_steps(model, fixed_noise[:4].clone(), schedule, seq, eta=0.0)
    print(f"  {n:>3} steps -- range: [{s.min():.2f}, {s.max():.2f}]")

# Visual comparison at very few steps
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for idx, n in enumerate([5, 10, 20, 50]):
    seq = uniform_timesteps(T, n)
    s = ddim_sample_custom_steps(model, fixed_noise[:4].clone(), schedule, seq, eta=0.0)
    grid = vutils.make_grid(denormalize(s), nrow=2, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[idx].set_title(f"{n} steps", fontsize=12)
    axes[idx].axis('off')

plt.suptitle("Minimum Steps for Recognizable Digits", fontsize=13)
plt.tight_layout()
plt.show()

---
## 7.5 -- Noise Schedules at Inference

The inference schedule typically matches the training schedule. The schedule values ($\bar{\alpha}_t$) at the selected timesteps are what matter during sampling -- the model was trained to predict noise at those specific noise levels.

When using DDIM with reduced steps, we simply index into the same schedule at the selected timestep positions. For example, with 50 uniformly-spaced steps from a T=1000 schedule, we use $\bar{\alpha}$ values at indices [0, 20, 40, ..., 980]. No rescaling is needed because DDIM's update rule directly uses $\bar{\alpha}_t$ and $\bar{\alpha}_{t-1}$.

**Practical guidance:**
- Use the same schedule for inference as for training
- For reduced-step DDIM, uniform sub-selection from the training schedule works well
- Mismatched schedules can cause artifacts (the model expects specific noise levels at each timestep)

In [ ]:
# Visualize which alpha_bar values are used at different step counts
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

alphas_cumprod_np = schedule['alphas_cumprod'].cpu().numpy()

# Left: Full schedule with sub-selected points
axes[0].plot(alphas_cumprod_np, color='lightgray', linewidth=1, label='Full schedule')
for n_steps, color in [(50, 'steelblue'), (20, 'darkorange'), (10, 'crimson')]:
    indices = torch.linspace(0, T - 1, n_steps, dtype=torch.long).numpy()
    axes[0].scatter(indices, alphas_cumprod_np[indices], s=15, color=color, label=f'{n_steps} steps', zorder=5)
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('alpha_bar_t')
axes[0].set_title('Sub-selected Schedule Points')
axes[0].legend(fontsize=9)

# Right: Uniform vs quadratic spacing
n = 20
uni = uniform_timesteps(T, n).numpy()
quad = quadratic_timesteps(T, n).numpy()
axes[1].stem(uni, alphas_cumprod_np[uni], linefmt='steelblue', markerfmt='o', basefmt=' ', label='Uniform')
axes[1].stem(quad, alphas_cumprod_np[quad], linefmt='darkorange', markerfmt='s', basefmt=' ', label='Quadratic')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('alpha_bar_t')
axes[1].set_title(f'Spacing Strategies ({n} steps)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 7.6 -- Implementation: Full Sampling Pipeline

A production sampling pipeline includes several practical details beyond the core algorithm:

1. **Noise generation**: Sample $x_T \sim \mathcal{N}(0, I)$
2. **Iterative denoising**: DDPM or DDIM loop
3. **Clamping**: Constrain values to $[-1, 1]$ (either at each step or only at the final step)
4. **Denormalization**: Map from $[-1, 1]$ to $[0, 1]$ for display
5. **Dynamic thresholding** (from [Imagen](https://arxiv.org/abs/2205.11487)): Instead of hard clamping at $[-1, 1]$, compute the p-th percentile of $|\hat{x}_0|$ and rescale, preventing saturation
6. **Batch generation**: Generate multiple samples efficiently

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: Full End-to-End Sampling Pipeline
# --------------------------------------------------------------------------

def dynamic_threshold(x0_pred: torch.Tensor, percentile: float = 0.995) -> torch.Tensor:
    """Dynamic thresholding from Imagen (Saharia et al. 2022).
    
    Instead of hard-clamping to [-1, 1], compute a data-dependent threshold
    and rescale. This prevents saturation in high-guidance settings.
    
    Args:
        x0_pred: (B, C, H, W) predicted clean image.
        percentile: Percentile for computing the threshold (default 99.5%).
    Returns:
        Thresholded tensor in [-1, 1].
    """
    B = x0_pred.shape[0]
    # Compute per-sample threshold
    flat = x0_pred.reshape(B, -1).abs()  # (B, C*H*W)
    threshold = torch.quantile(flat, percentile, dim=1, keepdim=True)  # (B, 1)
    threshold = torch.clamp(threshold, min=1.0)  # At least 1.0
    # Reshape for broadcasting
    threshold = threshold[:, :, None, None]  # (B, 1, 1, 1)
    # Clamp and rescale
    return torch.clamp(x0_pred, -threshold, threshold) / threshold  # (B, C, H, W)


@torch.no_grad()
def sampling_pipeline(
    model: nn.Module,
    num_samples: int,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    method: str = "ddim",
    num_steps: Optional[int] = None,
    eta: float = 0.0,
    use_dynamic_threshold: bool = False,
    clamp_each_step: bool = True,
    seed: Optional[int] = None,
    image_shape: Tuple[int, ...] = (1, 28, 28),
) -> torch.Tensor:
    """Complete sampling pipeline with all practical details.
    
    Args:
        model: Trained noise prediction model.
        num_samples: Number of images to generate.
        schedule: Pre-computed schedule dictionary.
        T: Total training timesteps.
        method: 'ddpm' or 'ddim'.
        num_steps: Sampling steps (None=T).
        eta: DDIM stochasticity (ignored for DDPM).
        use_dynamic_threshold: Apply dynamic thresholding to x0 predictions.
        clamp_each_step: Clamp x_t to [-1, 1] at each step.
        seed: Random seed for reproducibility.
        image_shape: (C, H, W) shape of each image.
    
    Returns:
        (num_samples, C, H, W) tensor in [0, 1] ready for display.
    """
    device = next(model.parameters()).device
    if seed is not None:
        torch.manual_seed(seed)
    
    shape = (num_samples, *image_shape)  # (B, C, H, W)
    
    if method == "ddpm":
        samples, _ = ddpm_sample(model, shape, schedule, T=T)
    elif method == "ddim":
        if num_steps is None:
            num_steps = T
        samples, _ = ddim_sample(model, shape, schedule, T=T, eta=eta, num_steps=num_steps)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Final clamp and denormalize
    if use_dynamic_threshold:
        samples = dynamic_threshold(samples)
    else:
        samples = samples.clamp(-1.0, 1.0)  # (B, C, H, W)
    
    samples = denormalize(samples)  # (B, C, H, W) in [0, 1]
    return samples


# Generate a 4x4 grid
samples = sampling_pipeline(
    model, num_samples=16, schedule=schedule, T=T,
    method='ddim', num_steps=50, eta=0.0, seed=42,
)
show_images(samples, nrow=4, title="Pipeline Output: 4x4 Grid (DDIM, 50 steps)")

In [ ]:
# Show the full denoising process for one sample as a row of images
torch.manual_seed(42)
trajectory_indices = list(range(0, 50, 5)) + [49]  # Every 5th step + final

_, trajectory = ddim_sample(
    model, shape=(1, 1, 28, 28), schedule=schedule, T=T,
    eta=0.0, num_steps=50,
    track_trajectory=True, trajectory_steps=trajectory_indices,
)

# Map trajectory indices to actual timestep values for display
step_indices = torch.linspace(0, T - 1, 50, dtype=torch.long).flip(0)
display_timesteps = [step_indices[i].item() for i in trajectory_indices if i < len(step_indices)]

if trajectory:
    show_denoising_trajectory(
        trajectory,
        timesteps=display_timesteps[:len(trajectory)],
    )

### Exercise 7.6: Full Pipeline with DDPM and DDIM

Use the `sampling_pipeline` function to generate and display samples with both DDPM (1000 steps) and DDIM (50 steps). Compare the visual quality and generation time.

In [ ]:
# Exercise 7.6: Your code here
# 1. Generate 16 samples with DDPM (method='ddpm', 1000 steps)
# 2. Generate 16 samples with DDIM (method='ddim', 50 steps, eta=0)
# 3. Measure wall-clock time for each
# 4. Display side by side

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# DDPM (full 1000 steps)
start = time.time()
samples_ddpm = sampling_pipeline(
    model, num_samples=16, schedule=schedule, T=T,
    method='ddpm', seed=42,
)
time_ddpm = time.time() - start

# DDIM (50 steps, deterministic)
start = time.time()
samples_ddim = sampling_pipeline(
    model, num_samples=16, schedule=schedule, T=T,
    method='ddim', num_steps=50, eta=0.0, seed=42,
)
time_ddim = time.time() - start

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

grid_ddpm = vutils.make_grid(samples_ddpm, nrow=4, padding=2)
axes[0].imshow(grid_ddpm.permute(1, 2, 0).cpu().numpy())
axes[0].set_title(f"DDPM (1000 steps, {time_ddpm:.1f}s)", fontsize=12)
axes[0].axis('off')

grid_ddim = vutils.make_grid(samples_ddim, nrow=4, padding=2)
axes[1].imshow(grid_ddim.permute(1, 2, 0).cpu().numpy())
axes[1].set_title(f"DDIM (50 steps, {time_ddim:.1f}s)", fontsize=12)
axes[1].axis('off')

plt.suptitle("DDPM vs DDIM: Quality and Speed", fontsize=14)
plt.tight_layout()
plt.show()

print(f"DDPM: {time_ddpm:.2f}s")
print(f"DDIM: {time_ddim:.2f}s")
print(f"Speedup: {time_ddpm / time_ddim:.1f}x")

---
## 7.7 -- Ancestral Sampling vs Probability Flow ODE

There are two equivalent perspectives on diffusion sampling that produce samples from the same distribution $p_\theta(x_0)$:

| Perspective | Type | Equation | Properties |
|:---|:---|:---|:---|
| Ancestral (SDE) | Stochastic | $dx = [f(x,t) - \frac{1}{2}g(t)^2 \nabla_x \log p_t(x)]dt + g(t)dw$ | Diverse samples, error correction |
| Probability Flow ODE | Deterministic | $dx = [f(x,t) - \frac{1}{2}g(t)^2 \nabla_x \log p_t(x)]dt$ | Reproducible, enables latent manipulation |

The score function relates to our noise prediction model: $\nabla_x \log p_t(x) \approx -\epsilon_\theta(x_t, t) / \sqrt{1 - \bar{\alpha}_t}$.

**DDIM with eta=0 is the Euler discretization of the probability flow ODE.** More advanced ODE solvers (Heun, DPM-Solver, DPM-Solver++) achieve better quality with fewer steps by using higher-order integration.

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: Probability Flow ODE via Manual Euler Method
# --------------------------------------------------------------------------

@torch.no_grad()
def probability_flow_ode_euler(
    model: nn.Module,
    shape: Tuple[int, ...],
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    num_steps: int = 1000,
    x_T: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Sample via Euler discretization of the probability flow ODE.
    
    The ODE in terms of alpha_bar:
        dx/d(alpha_bar) = (x - eps_theta) / (2 * alpha_bar)
                        - eps_theta / (2 * (1 - alpha_bar))
    
    We discretize this using the DDIM(eta=0) update, which IS the Euler method
    for this ODE. Here we implement it explicitly to show the connection.
    
    Args:
        model: Trained noise prediction model.
        shape: (B, C, H, W) sample shape.
        schedule: Pre-computed schedule dict.
        T: Total training timesteps.
        num_steps: Number of Euler steps.
        x_T: Optional initial noise.
    
    Returns:
        (B, C, H, W) generated samples.
    """
    device = next(model.parameters()).device
    
    if x_T is not None:
        x_t = x_T.clone()
    else:
        x_t = torch.randn(shape, device=device)  # (B, C, H, W)
    
    # Create timestep sequence
    step_indices = torch.linspace(0, T - 1, num_steps, dtype=torch.long)
    timesteps = step_indices.flip(0)  # high to low
    
    alphas_cumprod = schedule['alphas_cumprod']
    
    for i in range(len(timesteps)):
        t_val = timesteps[i].item()
        t_batch = torch.full((shape[0],), t_val, device=device, dtype=torch.long)
        
        # Get score estimate from noise prediction
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        alpha_bar_t = alphas_cumprod[t_val]
        if i < len(timesteps) - 1:
            alpha_bar_prev = alphas_cumprod[timesteps[i + 1].item()]
        else:
            alpha_bar_prev = torch.tensor(1.0, device=device)
        
        # DDIM(eta=0) update = Euler step of the probability flow ODE
        pred_x0 = ((x_t - torch.sqrt(1 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)).clamp(-1, 1)
        direction = torch.sqrt(1 - alpha_bar_prev) * eps_theta
        x_t = torch.sqrt(alpha_bar_prev) * pred_x0 + direction  # (B, C, H, W)
    
    return x_t


# Compare ODE Euler to DDIM(eta=0) -- they should match
torch.manual_seed(42)
fixed_noise = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

samples_ode = probability_flow_ode_euler(
    model, (4, 1, 28, 28), schedule, T=T, num_steps=50, x_T=fixed_noise.clone(),
)
samples_ddim_ref, _ = ddim_sample(
    model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=fixed_noise.clone(),
)

max_diff = (samples_ode - samples_ddim_ref).abs().max().item()
print(f"Max difference between ODE Euler and DDIM(eta=0): {max_diff:.8f}")
print("These are the same algorithm -- DDIM(eta=0) IS the Euler method for the probability flow ODE.")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
grid_ode = vutils.make_grid(denormalize(samples_ode), nrow=2, padding=2)
axes[0].imshow(grid_ode.permute(1, 2, 0).cpu().numpy())
axes[0].set_title("Probability Flow ODE (Euler)", fontsize=12)
axes[0].axis('off')

grid_ddim = vutils.make_grid(denormalize(samples_ddim_ref), nrow=2, padding=2)
axes[1].imshow(grid_ddim.permute(1, 2, 0).cpu().numpy())
axes[1].set_title("DDIM (eta=0)", fontsize=12)
axes[1].axis('off')

plt.suptitle("ODE Euler vs DDIM(eta=0): Equivalent", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 7.7: Euler ODE Sampling -- Quality and Speed Comparison

Implement the probability flow ODE with the Euler method and compare it to DDIM at various step counts. Measure both quality (visual) and speed (wall-clock time).

In [ ]:
# Exercise 7.7: Your code here
# 1. Use probability_flow_ode_euler with step counts [1000, 100, 50, 20, 10]
# 2. Compare to ddim_sample at the same step counts
# 3. Measure wall-clock time for each
# 4. Display grids and print timing table

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)
fixed_noise = torch.randn(8, 1, 28, 28, device=device)  # (8, 1, 28, 28)

step_counts = [100, 50, 20, 10]
ode_results = {}
ddim_results = {}
ode_times = {}
ddim_times = {}

for n_steps in step_counts:
    # ODE Euler
    start = time.time()
    ode_out = probability_flow_ode_euler(
        model, (8, 1, 28, 28), schedule, T=T, num_steps=n_steps, x_T=fixed_noise.clone(),
    )
    ode_times[n_steps] = time.time() - start
    ode_results[n_steps] = ode_out
    
    # DDIM eta=0
    start = time.time()
    ddim_out, _ = ddim_sample(
        model, (8, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=n_steps, x_T=fixed_noise.clone(),
    )
    ddim_times[n_steps] = time.time() - start
    ddim_results[n_steps] = ddim_out

# Display comparison
fig, axes = plt.subplots(2, len(step_counts), figsize=(3.5 * len(step_counts), 7))

for idx, n_steps in enumerate(step_counts):
    grid_ode = vutils.make_grid(denormalize(ode_results[n_steps]), nrow=4, padding=2)
    axes[0, idx].imshow(grid_ode.permute(1, 2, 0).cpu().numpy())
    axes[0, idx].set_title(f"ODE {n_steps} steps\n({ode_times[n_steps]:.2f}s)", fontsize=10)
    axes[0, idx].axis('off')
    
    grid_ddim = vutils.make_grid(denormalize(ddim_results[n_steps]), nrow=4, padding=2)
    axes[1, idx].imshow(grid_ddim.permute(1, 2, 0).cpu().numpy())
    axes[1, idx].set_title(f"DDIM {n_steps} steps\n({ddim_times[n_steps]:.2f}s)", fontsize=10)
    axes[1, idx].axis('off')

axes[0, 0].set_ylabel("ODE Euler", fontsize=12, rotation=0, labelpad=60)
axes[1, 0].set_ylabel("DDIM(eta=0)", fontsize=12, rotation=0, labelpad=60)
plt.suptitle("Probability Flow ODE (Euler) vs DDIM(eta=0)", fontsize=14)
plt.tight_layout()
plt.show()

# Print timing table
print(f"{'Steps':>6} | {'ODE (s)':>8} | {'DDIM (s)':>9} | {'Max Diff':>10}")
print("-" * 42)
for n in step_counts:
    diff = (ode_results[n] - ddim_results[n]).abs().max().item()
    print(f"{n:>6} | {ode_times[n]:>8.3f} | {ddim_times[n]:>9.3f} | {diff:>10.6f}")

print("\nAs expected, ODE Euler and DDIM(eta=0) produce identical results.")
print("They are mathematically the same algorithm expressed differently.")

---
## Capstone Exercise: Comprehensive DDPM vs DDIM Comparison

This exercise ties together everything from the module. Build a comprehensive comparison between DDPM and DDIM samplers:

1. Generate 64 samples with each method
2. Test DDIM at 1000, 100, 50, 20, and 10 steps
3. Compare visual quality, diversity, and wall-clock time
4. Show denoising trajectories for both methods
5. Demonstrate deterministic DDIM: same seed produces same image
6. Demonstrate stochastic DDPM: same seed produces different images (due to per-step noise)

In [ ]:
# Capstone Exercise: Your code here

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 1: Generate 64 samples with DDPM and DDIM

print("=" * 60)
print("CAPSTONE: Comprehensive DDPM vs DDIM Comparison")
print("=" * 60)

# DDPM: 64 samples, full 1000 steps
start = time.time()
samples_ddpm_64 = sampling_pipeline(
    model, num_samples=64, schedule=schedule, T=T,
    method='ddpm', seed=42,
)
time_ddpm_64 = time.time() - start
print(f"\nDDPM (1000 steps): {time_ddpm_64:.1f}s for 64 samples")

# DDIM: 64 samples, 50 steps, deterministic
start = time.time()
samples_ddim_64 = sampling_pipeline(
    model, num_samples=64, schedule=schedule, T=T,
    method='ddim', num_steps=50, eta=0.0, seed=42,
)
time_ddim_64 = time.time() - start
print(f"DDIM (50 steps):   {time_ddim_64:.1f}s for 64 samples")
print(f"Speedup:           {time_ddpm_64 / time_ddim_64:.1f}x")

# Display both grids
show_images(samples_ddpm_64, nrow=8, title="DDPM Samples (64, 1000 steps)")
show_images(samples_ddim_64, nrow=8, title="DDIM Samples (64, 50 steps, eta=0)")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 2: DDIM at different step counts

print("\n" + "=" * 60)
print("DDIM: Step Count Comparison")
print("=" * 60)

ddim_step_counts = [1000, 100, 50, 20, 10]
ddim_samples = {}
ddim_timings = {}

for n_steps in ddim_step_counts:
    start = time.time()
    samples = sampling_pipeline(
        model, num_samples=16, schedule=schedule, T=T,
        method='ddim', num_steps=n_steps, eta=0.0, seed=42,
    )
    elapsed = time.time() - start
    ddim_samples[n_steps] = samples
    ddim_timings[n_steps] = elapsed
    print(f"  {n_steps:>5} steps: {elapsed:.2f}s")

fig, axes = plt.subplots(1, len(ddim_step_counts), figsize=(3.5 * len(ddim_step_counts), 4))
for idx, n_steps in enumerate(ddim_step_counts):
    grid = vutils.make_grid(ddim_samples[n_steps], nrow=4, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[idx].set_title(f"{n_steps} steps\n({ddim_timings[n_steps]:.1f}s)", fontsize=11)
    axes[idx].axis('off')

plt.suptitle("DDIM Quality vs Step Count", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 3: Denoising trajectories

print("\n" + "=" * 60)
print("Denoising Trajectories")
print("=" * 60)

# DDPM trajectory
torch.manual_seed(42)
traj_steps_ddpm = [999, 750, 500, 250, 100, 50, 10, 0]
_, traj_ddpm = ddpm_sample(
    model, shape=(1, 1, 28, 28), schedule=schedule, T=T,
    track_trajectory=True, trajectory_steps=traj_steps_ddpm,
)
print(f"DDPM trajectory: {len(traj_ddpm)} snapshots")
show_denoising_trajectory(traj_ddpm, timesteps=traj_steps_ddpm[:len(traj_ddpm)])

# DDIM trajectory (50 steps)
torch.manual_seed(42)
traj_indices_ddim = [0, 5, 10, 15, 20, 30, 40, 49]
_, traj_ddim = ddim_sample(
    model, shape=(1, 1, 28, 28), schedule=schedule, T=T,
    eta=0.0, num_steps=50,
    track_trajectory=True, trajectory_steps=traj_indices_ddim,
)

step_indices = torch.linspace(0, T - 1, 50, dtype=torch.long).flip(0)
display_ts = [step_indices[i].item() for i in traj_indices_ddim if i < len(step_indices)]
print(f"DDIM trajectory: {len(traj_ddim)} snapshots")
if traj_ddim:
    show_denoising_trajectory(traj_ddim, timesteps=display_ts[:len(traj_ddim)])

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 4: Deterministic DDIM vs Stochastic DDPM

print("\n" + "=" * 60)
print("Determinism Test")
print("=" * 60)

torch.manual_seed(42)
shared_x_T = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

# DDIM (eta=0): Same seed -> same image (deterministic)
ddim_run1, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=shared_x_T.clone())
ddim_run2, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=shared_x_T.clone())
ddim_run3, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=shared_x_T.clone())

print(f"DDIM runs max diff (run1 vs run2): {(ddim_run1 - ddim_run2).abs().max().item():.8f}")
print(f"DDIM runs max diff (run1 vs run3): {(ddim_run1 - ddim_run3).abs().max().item():.8f}")
print("-> Deterministic: identical outputs from same x_T.")

# DDPM: Same x_T -> different images (stochastic per-step noise)
ddpm_run1 = ddpm_sample_from_noise(model, shared_x_T.clone(), schedule, T=T)
ddpm_run2 = ddpm_sample_from_noise(model, shared_x_T.clone(), schedule, T=T)
ddpm_run3 = ddpm_sample_from_noise(model, shared_x_T.clone(), schedule, T=T)

print(f"\nDDPM runs max diff (run1 vs run2): {(ddpm_run1 - ddpm_run2).abs().max().item():.4f}")
print(f"DDPM runs max diff (run1 vs run3): {(ddpm_run1 - ddpm_run3).abs().max().item():.4f}")
print("-> Stochastic: different outputs even from same x_T.")

# Visual comparison
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for i, (run, label) in enumerate([
    (ddim_run1, "DDIM run 1"), (ddim_run2, "DDIM run 2"), (ddim_run3, "DDIM run 3"),
]):
    grid = vutils.make_grid(denormalize(run), nrow=2, padding=2)
    axes[0, i].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[0, i].set_title(label, fontsize=11)
    axes[0, i].axis('off')

for i, (run, label) in enumerate([
    (ddpm_run1, "DDPM run 1"), (ddpm_run2, "DDPM run 2"), (ddpm_run3, "DDPM run 3"),
]):
    grid = vutils.make_grid(denormalize(run), nrow=2, padding=2)
    axes[1, i].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[1, i].set_title(label, fontsize=11)
    axes[1, i].axis('off')

plt.suptitle("Same x_T: DDIM is deterministic, DDPM is stochastic", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 5: Summary table

print("\n" + "=" * 60)
print("Summary")
print("=" * 60)

print(f"\n{'Method':<18} | {'Steps':>5} | {'Time (s)':>9} | {'Deterministic':>14}")
print("-" * 55)
print(f"{'DDPM':<18} | {'1000':>5} | {time_ddpm_64:>9.1f} | {'No':>14}")
for n_steps in ddim_step_counts:
    det = 'Yes' if True else 'No'
    print(f"{'DDIM (eta=0)':<18} | {n_steps:>5} | {ddim_timings[n_steps]:>9.2f} | {det:>14}")

print("\nKey Takeaways:")
print("- DDPM requires 1000 steps and is stochastic (diverse but slow).")
print("- DDIM(eta=0) is deterministic: same noise -> same image, enabling interpolation.")
print("- DDIM with 50 steps gives near-DDPM quality at ~20x speedup.")
print("- Both sample from the same learned distribution p_theta(x_0).")
print("- The probability flow ODE perspective unifies these approaches.")